**Load Fma Small dataset**

In [ ]:
from datasets import load_dataset, Dataset, Audio
from google.colab import drive
import os
from pathlib import Path
import json
from pydub import AudioSegment
from pydub.playback import play

In [ ]:
%pip install -Uq pydub

In [ ]:
!pip install 'torchcodec>=0.9.1'

In [ ]:
!pip uninstall -y datasets
!pip install datasets

import datasets
print(f"Current datasets version: {datasets.__version__}")

Found existing installation: datasets 3.6.0
Uninstalling datasets-3.6.0:
  Successfully uninstalled datasets-3.6.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 26.0 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0


Current datasets version: 3.6.0


In [ ]:

# Rock genre mappings
ROCK_GENRES = {
    130: "Rock",
    77: "Indie-Rock",
    88: "Krautrock",
    92: "Loud-Rock",
    107: "Noise-Rock",
    117: "Post-Punk",
    118: "Post-Rock",
    119: "Power-Pop",
    120: "Progressive",
    122: "Psych-Rock",
    123: "Punk",
    131: "Rock Opera",
    132: "Rockabilly",
    135: "Shoegaze",
    146: "Space-Rock",
    151: "Surf",
    157: "Thrash"
}

In [ ]:
print("Loading FMA Small dataset from Hugging Face...")
dataset = load_dataset("benjamin-paine/free-music-archive-small", split="train")
print(f"Total tracks in dataset: {len(dataset)}")

# Check what columns are available
print("\nDataset columns:", dataset.column_names)


Loading FMA Small dataset from Hugging Face...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Total tracks in dataset: 7916

Dataset columns: ['audio', 'title', 'url', 'artist', 'composer', 'lyricist', 'publisher', 'genres', 'tags', 'released', 'language', 'listens', 'artist_url', 'artist_website', 'album_title', 'album_url', 'license', 'copyright', 'explicit', 'instrumental', 'allow_commercial_use', 'allow_derivatives', 'require_attribution', 'require_share_alike']


In [ ]:
try:
    import torch
    print(f"torch version: {torch.__version__}")
except ImportError:
    print("torch not installed.")

try:
    import librosa
    print(f"librosa version: {librosa.__version__}")
except ImportError:
    print("librosa not installed.")

try:
    import torchcodec
    print(f"torchcodec version: {torchcodec.__version__}")
except ImportError:
    print("torchcodec not installed.")

try:
    import torchaudio
    print(f"torchaudio version: {torchaudio.__version__}")
except ImportError:
    print("torchaudio not installed.")

try:
    import datasets
    print(f"datasets version: {datasets.__version__}")
except ImportError:
    print("datasets not installed.")

torch version: 2.9.0+cu126
librosa version: 0.11.0
torchcodec version: 0.9.1
torchaudio version: 2.9.0+cu126
datasets version: 4.0.0


In [ ]:
print("\nSample entry:", dataset.take(2))


Sample entry: Dataset({
    features: ['audio', 'title', 'url', 'artist', 'composer', 'lyricist', 'publisher', 'genres', 'tags', 'released', 'language', 'listens', 'artist_url', 'artist_website', 'album_title', 'album_url', 'license', 'copyright', 'explicit', 'instrumental', 'allow_commercial_use', 'allow_derivatives', 'require_attribution', 'require_share_alike'],
    num_rows: 2
})


In [ ]:
sound = AudioSegment.from_file(dataset[0]['audio'])

AttributeError: 'AudioDecoder' object has no attribute 'read'

In [ ]:
from datasets import Audio
import io

# Get the current sampling rate from the dataset's audio feature
current_sampling_rate = dataset.features['audio'].sampling_rate

# Create a new Audio feature with decode=False
new_audio_feature = Audio(sampling_rate=current_sampling_rate, decode=False)

# Cast the 'audio' column to this new feature
dataset = dataset.cast_column('audio', new_audio_feature)

# Access the audio data
audio_data = dataset[0]['audio']

# Load using bytes if available, otherwise try path
if audio_data.get('bytes'):
    sound = AudioSegment.from_file(io.BytesIO(audio_data['bytes']))
else:
    sound = AudioSegment.from_file(audio_data['path'])

print("Audio loaded successfully!")

Audio loaded successfully!


In [ ]:
print("\nDataset columns:", dataset.column_names)


Dataset columns: ['audio', 'title', 'url', 'artist', 'composer', 'lyricist', 'publisher', 'genres', 'tags', 'released', 'language', 'listens', 'artist_url', 'artist_website', 'album_title', 'album_url', 'license', 'copyright', 'explicit', 'instrumental', 'allow_commercial_use', 'allow_derivatives', 'require_attribution', 'require_share_alike']


In [ ]:
sound[:current_sampling_rate]

In [ ]:
try:
    import torchcodec
    print(f"torchcodec successfully imported. Version: {torchcodec.__version__}")
except ImportError:
    print("Error: torchcodec is still not installed or not found.")


torchcodec successfully imported. Version: 0.9.1


In [ ]:
def is_rock_track(example):
    """
    Filter function to identify rock tracks.
    Adjust the field name based on actual dataset structure.
    """
    # Check different possible genre field names
    genre_field = None
    for field in ['genre_id', 'genre', 'genres', 'track_genre_top']:
        if field in example:
            genre_field = field
            break

    if genre_field is None:
        return False

    genre_value = example[genre_field]

    # Handle different genre formats
    if isinstance(genre_value, int):
        return genre_value in ROCK_GENRES
    elif isinstance(genre_value, list):
        return any(g in ROCK_GENRES for g in genre_value)
    elif isinstance(genre_value, str):
        # If genre is stored as string name
        return genre_value in ROCK_GENRES.values()

    return False

print("\nFiltering for rock genres...")
rock_dataset = dataset.filter(is_rock_track)
print(f"Found {len(rock_dataset)} rock tracks")


Filtering for rock genres...


Filter:   0%|          | 0/7916 [00:00<?, ? examples/s]

Found 872 rock tracks


# MVSep-MDX23 Colab Fork v2.5
Adaptation of MVSep-MDX23 algorithm for Colab, with few tweaks:

https://colab.research.google.com/github/jarredou/MVSEP-MDX23-Colab_v2/blob/v2.4/MVSep-MDX23-Colab.ipynb  
<br>  

Recent changes:  


**v2.5**
* Kim's MelBand-Roformer model added  


**v2.4**
* BS-Roformer models from viperx added
* MDX-InstHQ4 model added as optionnal
* Flac output
* Control input volume gain
* Filter vocals below 50Hz option
* Better chunking algo (no clicks)
* Some code cleaning

</font>
<br>

<details>
    <summary>Full changelog :</summary>
<br>
<font size=2>
<br>

[**v2.3**](https://github.com/jarredou/MVSEP-MDX23-Colab_v2/tree/v2.3)
* HQ3-Instr model replaced by VitLarge23 (thanks to MVSep)
* Improved MDXv2 processing (thanks to Anjok)
* Improved BigShifts algo (v2)
* BigShifts processing added to MDXv3 & VitLarge
* Faster folder batch processing

[**v2.2.2**](https://github.com/jarredou/MVSEP-MDX23-Colab_v2/tree/v2.2)
* Improved MDXv3 chunking code (thanks to HymnStudio)
* D1581 demo model replaced by new InstVocHQ MDXv3 model.
<br>

**v2.2.1**
* Added custom weights feature
* Fixed some bugs
* Fixed input: you can use a file or a folder as input now
<br>

**v2.2**
* Added MDXv3 compatibility
* Added MDXv3 demo model D1581 in vocals stem multiband ensemble.
* Added VOC-FT Fullband SRS instead of UVR-MDX-Instr-HQ3.
* Added 2stems feature : output only vocals/instrum (faster processing)
* Added 16bit output format option
* Added "BigShift trick" for MDX models
* Added separated overlap values for MDX, MDXv3 and Demucs
* Fixed volume compensation fine-tuning for MDX-VOC-FT
<br>

[**v2.1 (by deton24)**](https://github.com/deton24/MVSEP-MDX23-Colab_v2.1)
* Updated with MDX-VOC-FT instead of Kim Vocal 2
<br>

[**v2.0**](https://github.com/jarredou/MVSEP-MDX23-Colab_v2/tree/2.0)
* Updated with new Kim Vocal 2 & UVR-MDX-Instr-HQ3 models
* Folder batch processing
* Fixed high frequency bleed in vocals
* Fixed volume compensation for MDX models
<br>
</font>
</details>
<br>

Credits:
* [ZFTurbo/MVSep](https://github.com/ZFTurbo/MVSEP-MDX23-music-separation-model)
* Models by [Demucs](https://github.com/facebookresearch/demucs), [Anjok](https://github.com/Anjok07/ultimatevocalremovergui), [Kimberley Jensen](https://github.com/KimberleyJensen), [aufr33](https://github.com/aufr33) & viperx
* Adaptation & tweaks by [jarredou](https://github.com/jarredou/MVSEP-MDX23-Colab_v2/)
</font>

In [ ]:
#@markdown #Installation
#@markdown *Run this cell to install MVSep-MDX23*
print('Installing... This will take between 1 and 15 minutes, depending of how crappy Colab currently is...')
%cd /content
!git clone -b v2.5 https://github.com/jarredou/MVSEP-MDX23-Colab_v2  &> /dev/null
%cd /content/MVSEP-MDX23-Colab_v2
print('Installing dependencies...')
!pip install -r requirements.txt &> /dev/null
print('Installation done !')

Installing... This will take between 1 and 15 minutes, depending of how crappy Colab currently is...
/content
/content/MVSEP-MDX23-Colab_v2
Installing dependencies...
Installation done !


In [ ]:
#@markdown #Gdrive connection
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### About settings:


<font size=2>

* **BigShifts :** Better quality/speed performance with values between 3 and 11, **BUT** 11 doesn't always give the best results. Think about it like seed, different values will give slightly different results.<br>
Higher values = longer processing.
</font>

<font size=2>

* **Overlap InstVoc/VitLarge :** No big advantage to use high values when BigShifts is already high. If you use BigShifts=1 (regular processing), you can use higher values like 8 or even 16.<br>
Higher values = longer processing.<br>
 *Same goes with overlap_VOCFT, but with values between 0 and 0.95*
</font>

<font size=2>

* **Weights :** How much importance the result from the given model will have in final results.
</font>


In [ ]:
#@markdown #Separation
from pathlib import Path
import glob

%cd /content/MVSEP-MDX23-Colab_v2

#@markdown ---
#@markdown #### separation config:
input = '/content/drive/MyDrive/input' #@param {type:"string"}
output_folder = '/content/drive/MyDrive/VibeShift' #@param {type:"string"}

output_format = 'FLAC' #@param ["PCM_16", "FLOAT", "FLAC"]
Separation_mode = 'Vocals/Instrumental' #@param ["Vocals/Instrumental", "4-STEMS"]
input_gain = 0 #@param [0, -3, -6] {type:"raw"}
restore_gain_after_separation = False #@param {type:"boolean"}
filter_vocals_below_50hz = False #@param {type:"boolean"}
#@markdown ___
##@markdown

#@markdown  ### Model config:

#@markdown  *Set BigShifts=1 to disable that feature*
BigShifts = 3 #@param {type:"slider", min:1, max:41, step:1}
#@markdown ---
BSRoformer_model = 'ep_317_1297' #@param ["ep_317_1297", "ep_368_1296"]
weight_BSRoformer = 10 #@param {type:"slider", min:0, max:10, step:1}
##@markdown ---
weight_InstVoc = 4 #@param {type:"slider", min:0, max:10, step:1}
#@markdown ---
use_VitLarge = True #@param {type:"boolean"}
weight_VitLarge = 1 #@param {type:"slider", min:0, max:10, step:1}
#@markdown ---
use_InstHQ4 = False #@param {type:"boolean"}
weight_InstHQ4 = 2 #@param {type:"slider", min:0, max:10, step:1}
overlap_InstHQ4 = 0.1 #@param {type:"slider", min:0, max:0.95, step:0.05}
#@markdown ---
use_VOCFT = False #@param {type:"boolean"}
weight_VOCFT = 2 #@param {type:"slider", min:0, max:10, step:1}
overlap_VOCFT = 0.1 #@param {type:"slider", min:0, max:0.95, step:0.05}
#@markdown ---
#@markdown  *Demucs is only used in 4-STEMS mode.*
overlap_demucs = 0.6 #@param {type:"slider", min:0, max:0.95, step:0.05}

use_InstVoc_ = '--use_InstVoc' #forced use
use_BSRoformer_ =  '--use_BSRoformer' #forced use
use_VOCFT_ = '--use_VOCFT' if use_VOCFT is True else ''
use_VitLarge_ = '--use_VitLarge' if use_VitLarge is True else ''
use_InstHQ4_ = '--use_InstHQ4' if use_InstHQ4 is True else ''
restore_gain = '--restore_gain' if restore_gain_after_separation is True else ''
vocals_only = '--vocals_only' if Separation_mode == 'Vocals/Instrumental' else ''
filter_vocals = '--filter_vocals' if filter_vocals_below_50hz is True else ''

if Path(input).is_file():
  file_path = input
  Path(output_folder).mkdir(parents=True, exist_ok=True)
  !python inference.py \
        --input_audio "{file_path}" \
        --large_gpu \
        --BSRoformer_model {BSRoformer_model} \
        --weight_BSRoformer {weight_BSRoformer} \
        --weight_InstVoc {weight_InstVoc} \
        --weight_InstHQ4 {weight_InstHQ4} \
        --weight_VOCFT {weight_VOCFT} \
        --weight_VitLarge {weight_VitLarge} \
        --overlap_demucs {overlap_demucs} \
        --overlap_VOCFT {overlap_VOCFT} \
        --overlap_InstHQ4 {overlap_InstHQ4} \
        --output_format {output_format} \
        --BigShifts {BigShifts} \
        --output_folder "{output_folder}" \
        --input_gain {input_gain} \
        {filter_vocals} \
        {restore_gain} \
        {vocals_only} \
        {use_VitLarge_} \
        {use_VOCFT_} \
        {use_InstHQ4_} \
        {use_InstVoc_} \
        {use_BSRoformer_}


else:
    file_paths = sorted(glob.glob(input + "/*"))
    Path(output_folder).mkdir(parents=True, exist_ok=True)
    
    for file_path in file_paths:
        if Path(file_path).is_file():  # Ensure it's a file
            print(f"Processing: {file_path}")
            !python inference.py \
                --input_audio "{file_path}" \
                --large_gpu \
                --BSRoformer_model {BSRoformer_model} \
                --weight_BSRoformer {weight_BSRoformer} \
                --weight_InstVoc {weight_InstVoc} \
                --weight_InstHQ4 {weight_InstHQ4} \
                --weight_VOCFT {weight_VOCFT} \
                --weight_VitLarge {weight_VitLarge} \
                --overlap_demucs {overlap_demucs} \
                --overlap_VOCFT {overlap_VOCFT} \
                --overlap_InstHQ4 {overlap_InstHQ4} \
                --output_format {output_format} \
                --BigShifts {BigShifts} \
                --output_folder "{output_folder}" \
                --input_gain {input_gain} \
                {filter_vocals} \
                {restore_gain} \
                {vocals_only} \
                {use_VitLarge_} \
                {use_VOCFT_} \
                {use_InstHQ4_} \
                {use_InstVoc_} \
                {use_BSRoformer_}
            print(f"Completed: {file_path}\n")

/content/MVSEP-MDX23-Colab_v2
Error: The input path '000002.mp3' does not exist. Please check the path and try again.


In [ ]:
print(dataset[0]['audio']['path'])

000002.mp3


In [ ]:
import os
from pathlib import Path

# Your input file path from the previous cell
input_file_path = '/content/drive/MyDrive/VibeShift/livingOnAPrayer.mp3'

print(f"Checking accessibility of: {input_file_path}")

# 1. Check if Google Drive is mounted
if os.path.exists('/content/drive/MyDrive'):
    print("Google Drive appears to be mounted.")
else:
    print("WARNING: Google Drive does NOT appear to be mounted. Please run the 'Gdrive connection' cell (SEYsLSK0xd4-) first.")

# 2. Check if the parent directory exists and list its contents
parent_dir = Path(input_file_path).parent
if parent_dir.exists():
    print(f"\nParent directory '{parent_dir}' exists. Listing its contents:")
    !ls -lh "{parent_dir}"
else:
    print(f"\nError: Parent directory '{parent_dir}' does not exist.")

# 3. Check the file itself
if Path(input_file_path).exists():
    print(f"\nSuccess: File '{input_file_path}' exists and is accessible.")
    !stat "{input_file_path}" # Display file details
else:
    print(f"\nError: File '{input_file_path}' does NOT exist or is not accessible at this path.")


Checking accessibility of: /content/drive/MyDrive/VibeShift/livingOnAPrayer.mp3
Google Drive appears to be mounted.

Parent directory '/content/drive/MyDrive/VibeShift' exists. Listing its contents:
total 43M
-rw-r--r-- 1 root root  19M Jan 17 19:52 'Lime_-_Bon_Jovi_-_Livin_On_A_Prayer_(mp3.pm)_instrum.flac'
-rw-r--r-- 1 root root 9.5M Jan 17 19:48 'Lime_-_Bon_Jovi_-_Livin_On_A_Prayer_(mp3.pm).mp3'
-rw-r--r-- 1 root root  14M Jan 17 19:52 'Lime_-_Bon_Jovi_-_Livin_On_A_Prayer_(mp3.pm)_vocals.flac'

Error: File '/content/drive/MyDrive/VibeShift/livingOnAPrayer.mp3' does NOT exist or is not accessible at this path.
